# Periodic Splines from Fourier Coefficients
We display in thick gray the realization of a random periodic polynomial spline of a specified period $K,$ degree $n,$ and delay $\delta x;$ the red stemlines indicate the extent of one period. This periodic spline possesses an infinite-length series of Fourier coefficients that can be summarized without loss by the combination of, on one hand, a finite-dimensional vector $\hat{\mathbf{f}}$ of the first $\left\lfloor K/2\right\rfloor$ complex-valued Fourier coefficients of nonnegative indices and, on the other hand, of the three ancillary parameters $\{K,n,\delta x\}.$

We manipulate $\hat{\mathbf{f}}$ by setting a few of its components to zero. This corresponds to the application of a band-pass filter to the spline. The manipulated finite-dimensional vector of spline coefficients is then used to virtually reconstruct an infinite series of spline coefficients, and from them a spline, which we plot in blue.

It is sometimes desirable to handle in independent fashion the average value of the reconstructed spline. Here, the field ``keep_dc`` takes precedence over the specification of the bandpass range.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 20 # Maximal period
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic spline
s0 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(9), degree = 3)

# Plot
def update_plot (
    period = 9,
    degree = 3,
    delay = 0.0,
    bandpass = (9 // 6, 18 // 6),
    keep_dc = False
):
    global s0

    # Update of the spline
    if s0.period != period:
        s0 = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period),
            degree = s0.degree
        )
    s0.degree = degree
    s0.delay = delay

    # Fourier series
    f0 = s0.fourier_series()
    erase = f0.copy()
    erase[bandpass_widget.value[0] : bandpass_widget.value[1] + 1] = 0
    erase[0] = 0 if keep_dc else f0[0]
    s = sk.PeriodicSpline1D.from_fourier_series(
        f0 - erase,
        period = period,
        degree = degree,
        delay = delay
    )

    # Dynamic range
    image = {s0.image(), s.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    if isinstance(plotrange, sk.interval.Singleton):
        plotrange = sk.interval.Closed((
            plotrange.midpoint - 0.5,
            plotrange.midpoint + 0.5
        ))
    else:
        plotrange = sk.interval.Closed((
            plotrange.midpoint - 0.55 * plotrange.diameter,
            plotrange.midpoint + 0.55 * plotrange.diameter
        ))

    # Plot of the splines
    subplot = plt.subplots()
    s0.plot(
        subplot,
        plotpoints = 200 + 1,
        plotrange = plotrange,
        curve_fmt = "#E0E0E0",
        curve_lw = 7.0,
        curve_markerfmt = "oC7",
        curvestem_linefmt = "-C7",
        knot_marker = " "
    )
    s.plot(subplot, plotpoints = 200 + 1, plotrange = plotrange)
    # Final display
    plt.show()

# Interactions
period_widget = widgets.IntSlider(
    value = s0.period,
    min = 1,
    max = max_period,
    description = "period"
)
bandpass_widget = widgets.IntRangeSlider(
    value = (s0.period // 6, (2 * s0.period) // 6),
    min = 0,
    max = s0.period // 2,
    description = "bandpass"
)
def period_change (
    change
):
    bandpass_widget.max = change.new // 2
period_widget.observe(period_change, "value")
widgets.interactive(
    update_plot,
    period = period_widget,
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    bandpass = bandpass_widget,
    keep_dc = False
)
